In [0]:
dbutils.widgets.dropdown("src", "appointments", ["appointments","lab_results","medical_history","medication","patient","symptoms"])


In [0]:
src_value = dbutils.widgets.get("src")
src_value

In [0]:
src_values = ["appointments", "lab_results", "medical_history", "medication", "patient", "symptoms"]
dfs = []
for src_value in src_values:
    df = spark.readStream.format("cloudFiles")  \
          .option("cloudFiles.format","csv")  \
          .option("cloudFiles.schemaLocation", f"/Volumes/project/bronze/healthcare/{src_value}/checkpoint")  \
          .option("cloudFiles.schemaEvolutionMode", "rescue")   \
          .load(f"/Volumes/project/raw/raw/raw_hc/{src_value}/")
    dfs.append(df)

In [0]:
for src_value, df in zip(src_values, dfs):
    df.writeStream.format("delta")  \
        .outputMode("append") \
        .trigger(once=True) \
        .option("checkpointLocation",f"/Volumes/project/bronze/healthcare/{src_value}/checkpoint") \
        .option("path", f"/Volumes/project/bronze/healthcare/{src_value}/data") \
        .option("mergeSchema", "true")  \
        .start()